# How to Run
1. Begin by setting the granularity to 'q'. (change the accessed element in the list to 0)
2. Set run_every_query to True
3. Update max_quarter and max_month
4. Run All
5. Set the granularity to 'm'. (change the accessed element in the list to 1)
6. Set run_every_query to False and build_ms_df_from_scratch to True
7. Restart Kernel and Run All

In [10]:
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt
tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)
import openpyxl
import datetime as dt
import re
import os
import copy

In [23]:
granularity = ['q', 'm','w'][0]
run_every_query = False
max_quarter, max_month = 2, 6
max_week = dt.date.today().strftime("%U")
max_week = int(max_week)

In [12]:
def vintage_to_float(vintage):
    """
    Converts a vintage of form 'YYYY QQ' (i.e. '2016 Q2') to a float (i.e. 2016.25)
    The vectorized form of this function for a Series is: pd.to_numeric(df.vintage.str[:4]) + (pd.to_numeric(df.vintage.str[-1]) - 1) / 4
    NOTE: The following formula has been modified to allow for vintages of the form 'YYYY MMM' (i.e. '2016 M06')
    """
    if vintage[5]=='Q':
        return int(vintage[:4]) + (int(vintage[-1]) - 1) / 4
    elif vintage[5]=='M':
        return int(vintage[:4]) + (int(vintage[-2:]) - 1) / 12
    

def float_to_vintage(vintage_float):
    """
    Converts a vintage as a float (i.e. 2016.25) to a str of form 'YYYY QQ' (i.e. '2016 Q2')
    The vectorized form of this function for a Series is: 
        df.vintage_float.astype(int).astype(str) + ' Q' + ((df.vintage_float % 1) * 4 + 1).astype(int).astype(str)
    """
    return str(int(vintage_float)) + ' Q' + str(int((vintage_float % 1) * 4 + 1))
    
"""
SQL and Files
"""
def run_sql(filename, sub_list=[], connection=None, filename_is_query=False):
    """
    Run a SQL Query by reading from a .txt file, substituting values when required.
    Input:
    filename (str): File that we want to read. Usually a .txt file.
    sub_list (list of (str,str) tuples): Substitute each instance of the first element of the tuple for the second.
                                         Example: [('{max_mob}', '6')]
    """
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for sub in sub_list:
        text, var = sub
        query = query.replace(text, var)
#     print(query)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df

def store_pickle(data, filename):
    """
    Stores a pickle of the given data at the given filename, in a simplified function call
    """
    # Perform a swap if I got the order of data and filename backwards
    if type(data)==str:
        temp = filename
        filename = data
        data = temp
    with open(filename, 'wb') as file:
        pickle.dump(data, file)

def get_pickle(filename):
    """
    Loads a pickle in a simplified function call
    """
    with open(filename, 'rb') as file:
        data = pickle.load(file)
    return data

def smooth(series):
    """
    Calculates a centered average by averaging the previous two values and following two values, as well as the current value.
    Handles the first and last two values separately.
    """
    averaged_series = pd.Series(index=series.index, dtype=float)  # Create an empty Series to store the averaged values
    
    # Handle the first value separately
    averaged_series.iloc[0] = series.iloc[0]
    averaged_series.iloc[1] = (series.iloc[0] + series.iloc[1] + series.iloc[2]) / 3
    
    # Calculate the running average for the middle values
    for i in range(2, len(series) - 2):
        averaged_series.iloc[i] = (series.iloc[i-2] + series.iloc[i-1] + series.iloc[i] + series.iloc[i+1] + series.iloc[i+2]) / 5
    
    # Handle the last value separately
    averaged_series.iloc[-1] = series.iloc[-1]
    averaged_series.iloc[-2] = (series.iloc[-1] + series.iloc[-2] + series.iloc[-3]) / 3
    
    return averaged_series
def weight_by_proceeds(metric, proceeds):
    """
    Function to get the average of a given Series, weighted by Amount Financed.
    """
    return (metric * proceeds).sum() / proceeds.sum()
def weighted_average_and_sum(group, metrics):
    """
    Function that gets the weighted average of a Series (or several series), 
        returning the averages and the sum of Amount Financed.
    """
    if type(metrics)==str:
        weighted_avg = (group[metrics] * group.amt_financed).sum() / group.amt_financed.sum()
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        weighted_avg = (group[metric] * group.amt_financed).sum() / group.amt_financed.sum()
        result_dict[metric] = weighted_avg.sum()
    return pd.Series(result_dict)

# Old Recovery Model

In [13]:
def auc_pred(rra_df, leave_out, rra_lob_df=None):
    """
    Recovery model used by the pricing team to determine the expected recovery of a vehicle when it is sold.
    KMX uses a different recovery model, so it's handled separately.
    This function was made to support "Leave Out" analysis, which isn't included in the Ragu jar. 
        That's what all the if/else statements are for.
    This function has been vectorized as much as possible, but it's possibly not totally optimized.
    Inputs:
        rra_df (pd.DataFrame object): Data containing vehicle info. Each row is a vehicle, and each row is analyzed separately.
        leave_out (str): Factor to not consider in this runthrough. Helps us understand how much of a role each factor plays.
        rra_lob_df (pd.DataFrame): Only used in Leave Out analysis. Contains every vehicle for every vintage for this LOB.
    Outputs:
        recovery_multiplier (pd.Series): Column to add to rra_df containing the % of original vehicle value we expect to recover.
    """
    if len(rra_df[rra_df.lob=='KMX']) > 0:
        conditions = [
            (rra_df.car_age_orig < 3),
            (rra_df.car_age_orig == 3) & (rra_df.sale_price < 30000),
            (rra_df.car_age_orig == 3) & (30000 <= rra_df.sale_price) & (rra_df.sale_price < 40000),
            (rra_df.car_age_orig == 3) & (rra_df.sale_price >= 40000),
            (3 < rra_df.car_age_orig) & (rra_df.car_age_orig < 6) & (rra_df.sale_price < 30000),
            (3 < rra_df.car_age_orig) & (rra_df.car_age_orig < 6) & (30000 <= rra_df.sale_price) & (rra_df.sale_price < 40000),
            (3 < rra_df.car_age_orig) & (rra_df.car_age_orig < 6) & (rra_df.sale_price >= 40000),
            (rra_df.car_age_orig >= 6) & (rra_df.sale_price < 30000),
            (rra_df.car_age_orig >= 6) & (30000 <= rra_df.sale_price) & (rra_df.sale_price < 40000),
            (rra_df.car_age_orig >= 6) & (rra_df.sale_price >= 40000)
        ]

        values = [3300, 3700, 5400, 7200, 4700, 6400, 8200, 5700, 7400, 9200]

        recovery_amount = np.maximum(0, rra_df.sale_price - np.select(conditions, values, default=0))

        return recovery_amount * (1 - 0.206) ** rra_df.yob / rra_df.sale_price
    rra_df['coef_intercept'] = 0.0891185245
    rra_df['coef_yob'] = -0.2668087106 * rra_df.yob
    if leave_out != 'Mileage/age':
        rra_df['coef_mlg_x_age'] = 0.0001463508 * rra_df.car_age_orig * rra_df.mileage - 0.0004310013 * rra_df.car_age_orig - 0.0033127224 * rra_df.mileage
    else: 
        rra_df['coef_mlg_x_age'] = np.log((np.exp(0.0001463508 * rra_lob_df.car_age_orig * rra_lob_df.mileage) * np.exp(-0.0004310013 * rra_lob_df.car_age_orig) * np.exp(-0.0033127224 * rra_lob_df.mileage)).mean())
    if leave_out != 'Japanese':
        rra_df['coef_japanese'] = rra_df.car_japanese.map({'Japanese': 0.1004702010, 'Other': 0})
    else:
        rra_df['coef_japanese'] = np.log(rra_lob_df.car_japanese.map({'Japanese': np.exp(0.1004702010), 'Other': np.exp(0)}).mean())
    if leave_out != 'Vehicle class':
        rra_df['coef_class'] = rra_df.car_class_only.map({'Compact': 0.0303893787,
                                        'Minivan': np.log(0.975),
                                        'Sporty': 0.0758593720,
                                        'SUV Large': -0.0237364286,
                                        'SUV Small': 0.0190370746,
                                        'Work Van': 0,
                                        'Truck': 0.1436074633,
                                        'Car': 0, 'Other':0})
    else:
        rra_df['coef_class'] = np.log(rra_lob_df.car_class_only.map({'Compact': np.exp(0.0303893787),
                                                             'Minivan': np.exp(np.log(0.975)),
                                                             'Sporty': np.exp(0.0758593720),
                                                             'SUV Large': np.exp(-0.0237364286),
                                                             'SUV Small': np.exp(0.0190370746),
                                                             'Work Van': np.exp(0),
                                                             'Truck': np.exp(0.1436074633),
                                                             'Car': np.exp(0), 'Other':np.exp(0)}).mean())
    if leave_out != 'Fuel type':
        rra_df['coef_fuel'] = rra_df.fuel_type.map({'Hybrid': -0.0315481529,
                                       'Flex': 0.0255682747,
                                       'Diesel': -0.1284314925,
                                       'EV': np.log(0.9),
                                       'Gas': 0})
    else:
        rra_df['coef_fuel'] = np.log(rra_lob_df.fuel_type.map({'Hybrid': np.exp(-0.0315481529),
                                                    'Flex': np.exp(0.0255682747),
                                                    'Diesel': np.exp(-0.1284314925),
                                                    'EV': np.exp(np.log(0.9)),
                                                    'Gas': np.exp(0)}).mean())
    if leave_out != 'Luxury classification':
        rra_df['coef_lux'] = rra_df.car_lux.map({'Luxury': -0.1108317813, 'Standard': 0})
    else:
        rra_df['coef_lux'] = np.log(rra_lob_df.car_lux.map({'Luxury': np.exp(-0.1108317813), 'Standard': np.exp(0)}).mean())
    if leave_out != 'LOB':
        rra_df['coef_lob'] = rra_df.lob.map({'ENT': 0.0405731662,
                                      'FLD': 0.0405731662,
                                      'FRN': -0.0277901296,
                                      'STG': 0.0263749504,
                                      'AN': 0})
    else:
        rra_df['coef_lob'] = np.log(rra_lob_df.lob.map({'ENT': np.exp(0.0405731662),
                                              'FLD': np.exp(0.0405731662),
                                              'FRN': np.exp(-0.0277901296),
                                              'STG': np.exp(0.0263749504),
                                              'AN': np.exp(0)}).mean())
    if leave_out != 'Driver flag':
        rra_df['coef_driver'] = (-0.0918049638) * rra_df.driver_flag
    else:
        rra_df['coef_driver'] = np.log(rra_lob_df.driver_flag.map({1:np.exp(-0.0918049638), 0: np.exp(0)}).mean())
    if leave_out != 'Impound probability':
        rra_df['coef_impound'] = (-0.2803330880) * rra_df.impound_prob
    else:
        rra_df['coef_impound'] = np.log(np.mean(np.exp((-0.2803330880) * rra_lob_df.impound_prob)))
    
    try:
        # code that may raise an exception
        rra_df = rra_df.dropna()
        for col in ['coef_intercept', 'coef_yob', 'coef_mlg_x_age', 'coef_japanese', 'coef_class', 'coef_fuel', 'coef_lux', 
                    'coef_lux', 'coef_lob', 'coef_driver', 'coef_impound']:
            print(col)
            assert len(rra_df)==len(rra_df.dropna(subset=col))
    except Exception as e:
        # code to handle the exception
        print(col, ';', rra_df.fuel_type.unique())
        print(len(rra_df) - len(rra_df.fuel_type.dropna()))
        print(rra_df)
        store_pickle('rra_df_pickle_exception',rra_df)
        raise Exception()

    return rra_df.r_mmi * np.exp(rra_df.coef_intercept + rra_df.coef_yob + rra_df.coef_mlg_x_age + rra_df.coef_japanese\
                                 + rra_df.coef_class + rra_df.coef_fuel + rra_df.coef_lux + rra_df.coef_lob\
                                 + rra_df.coef_impound + rra_df.coef_driver)

In [14]:
def get_ula_multiplier_nonkmx(ula_df, leave_out):
    ula_df['loss_multiplier'] = 1
    if leave_out!='Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag # loss_adj_01_acaco: 1 or 1.1
    if leave_out!='Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag *(1- ula_df.pricing_change_flag) # loss_adj_02_small_amtfin: 1 or 1.1
    if leave_out!='Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag) # loss_adj_03_zero_down: 1 or 1.1
    if leave_out!='High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag # loss_adj_04_high_mileage: 1 or 1.1
    if leave_out!='High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag) # loss_adj_05_high_PTI: 1 or 1.1
    if leave_out!='Car make':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.car_make_penalty_flag\
                                    - 0.1 * ula_df.car_make_benefit_flag - 0.1*ula_df.pricing_change_flag*ula_df.car_make_benefit_flag
                                    #+  0.1*ula_df.pricing_change_flag*ula_df.car_make_benefit_flag*ula_df.ent_fld_flag # loss_adj_06_vehicle_make: 1 or 0.9 or 1.1
    if leave_out!='Theft risk':
        ula_df.loss_multiplier *= 0.987 + 0.099 * ula_df.theft_risk_flag * (1-ula_df.pricing_change_flag) + 0.013*ula_df.pricing_change_flag  # loss_adj_09_theft_risk: 0.987 or 1.086
    if leave_out!='MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag # loss_adj_07_new_MCY: 1 or 0.8
    if leave_out!='Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag\
                                    + 0.02 * ula_df.weekday_flag # loss_adj_08_day_of_week: 1 or 0.95 or 1.02
    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag 
        #ula_df.loss_multiplier *= 0.989 + 0.111 * ula_df.secured_credit_flag # loss_adj_10_secured_card: 0.989 or 1.1
    if leave_out!='Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag\
                                    - 0.1 * ula_df.waiter_employment_flag # loss_adj_11_employment: 1 + 0.1 * cust_attr$employment_flag
    if leave_out!='Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag # loss_adj_13_authorized_tradeline: 1 or 1.1
    if leave_out!='Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2) #previously (0.8,1.2)
    #if leave_out!='Null FICO with Vantage':
        #ula_df.loc[ula_df['fico_adjustment_flag'] == 1, 'loss_multiplier'] *= 0.99 + 0.11 * ula_df.null_fico_w_vantage_flag # loss_adj_14_non_fico: 0.99 or 1.1
    if leave_out!='Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4) #previously (0.8,1.2)
    # if leave_out!='Dealer Level (Non-KMX)':
    #     ula_df.loss_multiplier *= 1 - 0.2 * (ula_df.lob_or_bucket == 'A').astype(int) \
    #                                 - 0.1 * (ula_df.lob_or_bucket == 'B').astype(int) \
    #                                 + 0.09 * (ula_df.lob_or_bucket == 'D').astype(int) \
    #                                 + 0.2 * (ula_df.lob_or_bucket == 'E').astype(int)
        #change for lob dll ragu
    # The following is a pricing adjustment, but we don't include it, since it's macroeconimic (that's what the loss committee predicts)
#       #Additional loss adjustment to ensure losses match loss committe predictions
#   lob_scalar <- ifelse(app_attr$lob == "AN", 1.045,
#                        ifelse(app_attr$lob == "FLD", 1.09,
#                               ifelse(app_attr$lob == "FRN", 0.925,
#                                      ifelse(app_attr$lob == "STG", 0.996, 1))))
    return ula_df

In [15]:
def get_ula_multiplier_kmx(ula_df, loss_scale=0.067, leave_out='None'):
    ula_df['loss_multiplier'] = 1
    if leave_out!='Job time':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.2 * ula_df.job_time_flag
    if leave_out!='Low FICO':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_fico_flag & ~ula_df.high_model_score_flag)\
                                    + loss_scale * (ula_df.low_fico_flag & ula_df.high_model_score_flag)
    if leave_out!='Low Vantage':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.25 * (ula_df.low_vantage_flag & ~ula_df.high_model_score_flag)\
                                    + loss_scale * (ula_df.low_vantage_flag & ula_df.high_model_score_flag)
    if leave_out!='Louisiana':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.35 * ula_df.louisiana_flag
    if leave_out!='High PTI':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0 * ula_df.normal_pti_flag \
                                    + 0.05 * ula_df.high_pti_tier_1_flag \
                                    + 0.1 * ula_df.high_pti_tier_2_flag \
                                    + (1.4 * (1 + loss_scale) - 1) * ula_df.high_pti_tier_3_flag
    if leave_out!='Existing DQ':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.1 * ula_df.existing_dq_flag
    if leave_out!='Employment type':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] /= (1 + loss_scale)
    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + (-0.01 + 0.06 * ula_df.secured_credit_flag)\
                                        * ~(~ula_df.job_time_flag & ula_df.narrowed_soft_pull_flag & ula_df.secured_credit_flag)
    if leave_out!='Authorized tradelines':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 0.99 + 0.18 * ula_df.kmx_auth_tradelines_flag
    if leave_out!='Null FICO with Vantage':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 0.99 + 0.11 * ula_df.null_fico_w_vantage_flag + 0.01 * ula_df.null_fico_null_vantage_flag
    if leave_out!='Soft pull':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + ~ula_df.job_time_flag * (~ula_df.narrowed_soft_pull_flag * -0.025
                                                               + ula_df.narrowed_soft_pull_flag * (~ula_df.secured_credit_flag * 0.108
                                                                                                   + ula_df.secured_credit_flag * 0.295))
    if leave_out!='Clip':
        clipped_multiplier = np.clip(ula_df[~ula_df.high_pti_tier_3_flag].loss_multiplier, 0.8, 1.35 / (1 + loss_scale))
        ula_df.loc[~ula_df.high_pti_tier_3_flag & ~ula_df.mtn_3_1_flag , 'loss_multiplier'] = clipped_multiplier
    if leave_out!='Vehicle Age':
        ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier']  *= 0.8593 + 0.0201 * ula_df.continuous_vehicle_age
    
    
    #Mountain 3.1 Gross Loss Adjustments
    
    if leave_out!='Louisiana':
        ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier']  *= 1 + 0.35 * ula_df.louisiana_flag
    if leave_out!='Secured credit (Chime, etc.)':
        ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier'] *= 0.96 + (0.01 * ula_df.soft_pull_flag  - 0.18*ula_df.chime_flag* ula_df.soft_pull_flag) + 0.46*ula_df.chime_flag
                                    
    if leave_out!='Job time':
        ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier'] *= 0.99 + 0.21 * ula_df.job_time_flag
    if leave_out!='Existing DQ':
        ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier'] *= 0.99 + 0.11 * ula_df.existing_dq_flag
    if leave_out!='Employment type':
        ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.1 * ula_df.seasonal_employment_flag
    if leave_out!='Authorized tradelines':
        ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier'] *= 0.99 + 0.06 * ula_df.kmx_auth_tradelines_flag
    
    if leave_out!='Soft pull':
        ula_df.loc[ula_df.mtn_3_1_flag & ula_df.soft_pull_flag,'loss_multiplier'] *= 1.1  * (0.99 + 0.11*ula_df.low_bureau_flag) * (0.97 + 0.15*ula_df.cd_perc_flag) * (0.978 + 0.172*ula_df.open_tl_flag)
        ula_df.loc[ula_df.mtn_3_1_flag & ~ula_df.soft_pull_flag,'loss_multiplier'] *= 1 * (0.98 + 0.22*ula_df.low_bureau_flag)* (0.954 + 0.346*ula_df.cd_perc_flag) * (0.945 + 0.405*ula_df.open_tl_flag )/np.maximum(ula_df.cd_perc_flag*ula_df.open_tl_flag*1.2,1)
    
    # if leave_out!='Soft pull':
    #     ula_df.loc[ula_df.mtn_3_1_flag & ula_df.soft_pull_flag,'loss_multiplier'] *= 1.1 * (0.97 + 0.23*ula_df.secured_credit_flag) * (0.995 + 0.03*ula_df.low_bureau_flag) * (0.97 + 0.15*ula_df.cd_perc_flag) * (0.978 + 0.172*ula_df.open_tl_flag)
    #     ula_df.loc[ula_df.mtn_3_1_flag & ~ula_df.soft_pull_flag,'loss_multiplier'] *= 1 * (0.98 + 0.32*ula_df.secured_credit_flag) * (0.9767 + 0.2233*ula_df.low_bureau_flag)* (0.954 + 0.346*ula_df.cd_perc_flag) * (0.945 + 0.405*ula_df.open_tl_flag )/np.maximum(ula_df.cd_perc_flag*ula_df.open_tl_flag*1.2,1)
    ula_df.loc[ula_df.mtn_3_1_flag,'loss_multiplier']/=1.1
    if leave_out!='Clip':
        clipped_multiplier = np.clip(ula_df[ula_df.mtn_3_1_flag].loss_multiplier, 0.75, 1.4)
        ula_df.loc[ula_df.mtn_3_1_flag , 'loss_multiplier'] = clipped_multiplier
    if leave_out!='Car make':
        ula_df.loss_multiplier *= 1 - 0.08*ula_df.kmx_toyho_flag
        
    return ula_df

In [16]:
def get_ragu_score(vintage, lob, lob_type, ula_df_total, rra_df_total, leave_out='None'):
    """
    This function provides the core of RAGU Score analysis.
    There are three parts to this code:
        1. Get Unit Loss Adjustments: Modify the probability of chargeoff based on factors not included in Model Score.
        2. Get Recovery Rate Adjustments: Use auc_pred() to calculate how much we'll recover if the vehicle sells.
        3. Group it all together: Group based on granularity, applying the RAGU Score calculation formulas (see PowerPoint).
            This section rolls up by LOB, but if there are multiple LOBs (for non_kmxent), group by that in addition.
    Inputs:
        vintage (str): Monthly or quarterly vintage (for example, "2022 M03" or "2022 Q1") we want results for
        lob (str or tuple of str): LOB we want results for. 
            If multiple are provided, they are assessed in the context of each other, but still separate.
        lob_type (str): Higher-order group. For KMX and ENT, this is the same as LOB, but for other LOBs, this is 'non_kmxent'
        ula_df_total (pd.DataFrame object): DataFrame containing every vehicle's ULA-related info throughout ACA history.
        rra_df_total (pd.DataFrame object): DataFrame containing every vehicle's RRA-related info throughout ACA history.
        leave_out (str): For Leave Out analysis, what factor to leave out. This helps us assess how much impact each factor has.
    Outputs:
        full_df (pd.DataFrame): DataFrame containing the results of RAGU Score analysis for the given vintage and LOB(s). 
            All columns are weighted averages unless otherwise noted.
            Columns: 'amt_financed_x': Sum of amount financed for this grouping
                     'loss_multiplier': Result of ULA.  What % more or less the unit loss rate should be.
                     'recovery_unadjusted_multiplier': % of original vehicle value we expect to recover, not adjusted for LTV or baseline.
                     'ltv': LTV of grouping
                     'bbvalue': BlackBook value of grouping. For KMX, this is sale price instead.
                     'ltv_realization_factor': Unused. Factor to determine how much benefit from LTV we're seeing.
                     'recovery_100_ltv_multiplier': Unused. recovery_unadjusted_multiplier adjusted for LTV, assuming 100% impact.
                     'recovery_multiplier': Unused. recovery_unadjusted_multiplier adjusted for LTV, after ltv_realization_factor.
                     'vintage': Vintage for this grouping.
                     'model_score': Model score for this grouping. Same as ms_original.
                     'amt_financed_y': Unused. Sum of amount financed for this grouping, using different parts of ms_df. Nearly the same as amt_financed_x.
                     'est_unit_loss': Probability of chargeoff. Currently set to mean_unit_loss. We may eventually modify this to convert the Unit Loss Score to a Unit Loss %.
                     'unit_loss_score': Model Score adjusted for ULA.
                     'ms_original': Model score for this grouping. Same as model_score (for the moment, this may change when we get better historical values for the current model).
                     'baselined_recovery': recovery_multiplier / baseline_recovery_pct. How much better/worse we're doing for this grouping compared to the grouping's baseline.
                     'baselined_unadjusted_recovery': recovery_unadjusted_multiplier / baseline_recovery_unadjusted_pct
                     'baselined_100_ltv_recovery': recovery_100_ltv_multiplier / baseline_recovery_100_ltv_pct
                     'ms_gla': Renamed Unit Loss Score
                     'ms_exclude_ltv': RAGU Score without any LTV impacts. Because we have some issues with LTV, this is the score we use, taking LTV adjustments in the Excel.
                     'ms_100_ltv': RAGU Score assuming 100% LTV impacts.
                     'ragu_score': RAGU Score using baselined_recovery. Uses the RAGU formulas.
    """
    # Get the Unit Loss Adjustment Multiplier
    if type(lob)==str:
        ula_df = ula_df_total[(ula_df_total.vintage==vintage)&(ula_df_total.lob==lob)].copy()
        ula_lob_df = ula_df_total[(ula_df_total.lob==lob)].copy()
    else:
        ula_df = ula_df_total[(ula_df_total.vintage==vintage)&(ula_df_total.lob.isin(lob))].copy()
        ula_lob_df = ula_df_total[(ula_df_total.lob.isin(lob))].copy()
    if lob == 'KMX':
        loss_scale = 0.067
        ula_df = get_ula_multiplier_kmx(ula_df, loss_scale, leave_out)
    else:
        ula_df = get_ula_multiplier_nonkmx(ula_df, leave_out)
    ula_df = ula_df[['account_number', 'book_date', 'bbvalue', 'sale_price', 'amt_financed', 'lob_or_bucket', 'lob', 'loss_multiplier']]
    # Get the Recovery Rate Adjustment Multiplier
    if leave_out == 'None':
        if type(lob)==str:
            rra_df = rra_df_total[(rra_df_total.vintage==vintage)&(rra_df_total.lob==lob)].copy()
        else:
            rra_df = rra_df_total[(rra_df_total.vintage==vintage)&(rra_df_total.lob.isin(lob))].copy()
    else:
        if type(lob)==str:
            rra_df = rra_df_total[(rra_df_total.lob==lob)].copy()
        else:
            rra_df = rra_df_total[(rra_df_total.lob.isin(lob))].copy()
    try:
        rra_df.car_year = rra_df.car_year.fillna(round(rra_df.car_year.mean()))
    except:
        store_pickle('rra_df_pickle_exception',rra_df)
    # Note: ceil(car_age_orig) = ula_df's vehicle_age column
    rra_df['car_age_orig'] = np.maximum((pd.to_datetime(rra_df.book_date) - pd.to_datetime(rra_df.car_year.astype(int).astype(str) + '-09-01')).dt.days / 365, 1 / 365)
    if leave_out == 'None':
        rra_df['recovery_multiplier'] = auc_pred(rra_df, leave_out)
    else:
        rra_lob_df = rra_df.copy()
        rra_df = rra_df[rra_df.vintage==vintage]
        rra_df['recovery_multiplier'] = auc_pred(rra_df, leave_out, rra_lob_df)
    rra_df = rra_df[['account_number', 'recovery_multiplier']]
    mix_df = ula_df.merge(rra_df, on='account_number', how='inner').drop_duplicates(subset='account_number', keep='first')
    bb_populated_df = mix_df.dropna(subset='bbvalue')
    bb_populated_df['ltv'] = bb_populated_df.amt_financed / bb_populated_df.bbvalue
    bb_populated_df['recovery_unadjusted_multiplier'] = skip_rate * bb_populated_df.recovery_multiplier.copy()
    # Group by LOB
    grouped_mix_df = bb_populated_df.groupby('lob').apply(weighted_average_and_sum, ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'])
    # Calculate these values after rollup to match the LTV graph better.
    grouped_mix_df['ltv_realization_factor'] = 0 # 1 - ltv_realization_dollar / (grouped_mix_df.recovery_unadjusted_multiplier * grouped_mix_df.bbvalue)
    grouped_mix_df['recovery_100_ltv_multiplier'] = 0 # (grouped_mix_df.recovery_unadjusted_multiplier * baseline_ltv / grouped_mix_df.ltv).copy()
    grouped_mix_df['recovery_multiplier'] = 0 # grouped_mix_df.recovery_unadjusted_multiplier * (1 + (baseline_ltv / grouped_mix_df.ltv - 1) * grouped_mix_df.ltv_realization_factor)
    vintage_ms_df = ms_df[ms_df.vintage==vintage].copy()
    #store_pickle('vintage_ms_df_keyerror', vintage_ms_df)
    #store_pickle('grouped_mix_df_keyerror', grouped_mix_df)
    vintage_ms_df = vintage_ms_df.reset_index(drop = True) #revert to original, remove line
    #grouped_mix_df = grouped_mix_df.reset_index(drop = True) #revert to original, remove line
    full_df = grouped_mix_df.merge(vintage_ms_df, on='lob')
    full_df['est_unit_loss'] = mean_unit_loss
    full_df['unit_loss_score'] = full_df.model_score + (1 - full_df.loss_multiplier) * full_df.est_unit_loss / unit_loss_to_model_score
    full_df = full_df.set_index('lob')
    # Step 1: Get Model Score
    full_df['ms_original'] = full_df.model_score.copy()
    # Step 2: Adjust Model Score by Pricing Adjustments (This includes Unit Loss and pre-LTV Recovery adjustments)
    full_df['baselined_recovery'] = 0 # (full_df.recovery_multiplier / baseline_recovery_pct).copy()
    full_df['baselined_unadjusted_recovery'] = (full_df.recovery_unadjusted_multiplier / baseline_recovery_unadjusted_pct).copy()
    full_df['baselined_100_ltv_recovery'] = 0 # (full_df.recovery_100_ltv_multiplier / baseline_recovery_100_ltv_pct).copy()
    if type(lob) != str and len(lob) > 1:
        full_df.loc[lob_type, 'unit_loss_score'] = weight_by_proceeds(full_df.unit_loss_score, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'est_unit_loss'] = mean_unit_loss
        full_df.loc[lob_type, 'ms_original'] = weight_by_proceeds(full_df.model_score, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'baselined_recovery'] = weight_by_proceeds(full_df.recovery_multiplier / baseline_recovery_pct, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'baselined_unadjusted_recovery'] = weight_by_proceeds(full_df.recovery_unadjusted_multiplier / baseline_recovery_unadjusted_pct, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'baselined_100_ltv_recovery'] = weight_by_proceeds(full_df.recovery_100_ltv_multiplier / baseline_recovery_100_ltv_pct, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'recovery_multiplier'] = weight_by_proceeds(full_df.recovery_multiplier, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'recovery_unadjusted_multiplier'] = weight_by_proceeds(full_df.recovery_unadjusted_multiplier, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'recovery_100_ltv_multiplier'] = weight_by_proceeds(full_df.recovery_100_ltv_multiplier, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'vintage'] = vintage
        full_df.loc[lob_type, 'ltv'] = weight_by_proceeds(full_df.ltv, full_df.amt_financed_x) # TODO: Determine whether _x or _y is correct
        full_df.loc[lob_type, 'amt_financed_x'] = full_df.amt_financed_x.sum()
    full_df['ms_gla'] = full_df.unit_loss_score.copy()
    full_df['ms_exclude_ltv'] = (1 - full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier) * full_df.unit_loss_score + full_df.est_unit_loss * full_df.recovery_unadjusted_multiplier * full_df.unit_loss_score * full_df.baselined_unadjusted_recovery
    full_df['ms_100_ltv'] = (1 - full_df.est_unit_loss * full_df.recovery_100_ltv_multiplier) * full_df.unit_loss_score + full_df.est_unit_loss * full_df.recovery_100_ltv_multiplier * full_df.unit_loss_score * full_df.baselined_100_ltv_recovery
    # Step 3: RAGU Score
    full_df['ragu_score'] = (1 - full_df.est_unit_loss * full_df.recovery_multiplier) * full_df.unit_loss_score + full_df.est_unit_loss * full_df.recovery_multiplier * full_df.unit_loss_score * full_df.baselined_recovery
    return full_df

## Fetch MS for a given bucket

In [24]:
desired_bucket = 'B'

In [25]:
"""
Refetch the original model scores, untainted by any modification by analysis.
"""
run_query = False
if run_query or run_every_query:
    all_original_model_scores = run_sql('postmodern_ms_query.txt', sub_list=[('{min_book_date}', "'2016-01-01'")])
    store_pickle(all_original_model_scores, 'all_original_model_scores_pickle')
else:
    all_original_model_scores = get_pickle('all_original_model_scores_pickle')
all_original_model_scores.book_date = all_original_model_scores.book_date.astype(str)
all_original_model_scores.book_week = all_original_model_scores.book_week.astype(str)
#print(all_original_model_scores.book_date)
if granularity == 'q':
    all_original_model_scores['vintage'] = all_original_model_scores.book_date.str[:4] + ' Q' + ((all_original_model_scores.book_date.str[5:7].astype(int) - 1) // 3 + 1).astype(str)
elif granularity == 'm':
    all_original_model_scores['vintage'] = all_original_model_scores.book_date.str[:4] + ' M' + all_original_model_scores.book_date.str[5:7]
else:
    all_original_model_scores['vintage'] = all_original_model_scores.book_date.str[:4] + '-'+ all_original_model_scores.book_week.str.zfill(2)
all_original_model_scores = all_original_model_scores[all_original_model_scores.apr_bucket==desired_bucket]    
original_model_scores = all_original_model_scores.groupby(['vintage', 'lob']).apply(weighted_average_and_sum, 'model_score').reset_index()
original_model_scores = pd.concat([original_model_scores, all_original_model_scores.groupby(['vintage']).apply(weighted_average_and_sum, 'model_score').reset_index()]).fillna('POS')
original_model_scores = pd.concat([original_model_scores, all_original_model_scores[all_original_model_scores.lob.isin(['AN', 'STG', 'FRN', 'FLD'])].groupby(['vintage']).apply(weighted_average_and_sum, 'model_score').reset_index()]).fillna('non_kmxent')
ms_df = original_model_scores.copy()

C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\64835687.py:20: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  original_model_scores = all_original_model_scores.groupby(['vintage', 'lob']).apply(weighted_average_and_sum, 'model_score').reset_index()
C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\64835687.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  original_model_scores = pd.concat([orig

In [26]:
ms_df.head() 

,vintage,lob,model_score,amt_financed
0,2016 Q1,AN,128.775121,27554664.67
1,2016 Q1,Core,134.582566,78255.03
2,2016 Q1,FLD,133.443172,405774.69
3,2016 Q1,FRN,128.640395,11426787.02
4,2016 Q1,KMX,131.355232,86442050.89


# Fetch ula_df_total, rra_df_total

In [27]:
"""
The following block of code establishes ula_df_total and rra_df_total, as well as some standard assumptions.
The SQL query takes a very long time (30+ minutes), since it's a series of temp tables and gets every single loan.
All the calculations after the query should be vectorized, speeding up the process.
"""
expected_years_on_book = 2
impound_probability = 0.15
mmi_standard_increase = 1.03
run_query = False
generate_dfs = True
if generate_dfs:
    if run_query or run_every_query:
        with pyodbc.connect("DSN=Redshift_prod") as conn:
            with open("establish_temp_tables_query.txt", "r") as file:
                temp_table_queries = file.read()
            conn.execute(temp_table_queries.strip())
            ula_df_total = run_sql('vintage_level_ula_query.txt', connection=conn)
            rra_df_total = run_sql('vintage_level_rra_query.txt', connection=conn)
            dla_df = run_sql('scratch_query.txt', connection = conn)
            print('tables query finished')
        store_pickle((ula_df_total, rra_df_total,dla_df), '(ula_df_total, rra_df_total, dla_df)_unrefined_pickle')
    else:
        ula_df_total, rra_df_total, dla_df = get_pickle('(ula_df_total, rra_df_total, dla_df)_unrefined_pickle')
    # Make DataFrame-wide changes
    rra_df_total = rra_df_total[rra_df_total.lob != 'Core'] # Core Model Scores are missing
    ula_df_total = ula_df_total[ula_df_total.lob != 'Core'] # Core Model Scores are missing
    #Filter out data for each bucket
    ula_df_total = ula_df_total[ula_df_total.apr_bucket==desired_bucket]
    rra_df_total = rra_df_total[rra_df_total.apr_bucket==desired_bucket]
    #MTN 3.1 flag
    ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
    #ula_df_total['mtn_3_1_flag'] = ula_df_total.app_date>= pd.to_datetime("2024-12-16")
    ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model== 'MTN3.1'
    rra_df_total.book_date = rra_df_total.book_date.astype(str)
    ula_df_total.book_date = ula_df_total.book_date.astype(str)
    rra_df_total.book_week = rra_df_total.book_week.astype(str)
    ula_df_total.book_week = ula_df_total.book_week.astype(str)
    ula_df_total['vehicle_age'] = np.maximum(ula_df_total.book_date.str[:4].astype(int) - ula_df_total.model_year, 1/365)
    ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
    ula_df_total.make = ula_df_total.make.str.upper().str[:3]
    ula_df_total.lob_or_bucket = np.select([ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
                                          ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])], 
                                         ['D', 'C'], default=ula_df_total.lob_or_bucket)
    ula_df_total['continuous_vehicle_age'] = ula_df_total.book_date.str[:4].astype(int) + ula_df_total.book_date.str[5:7].astype(int)/12 - (ula_df_total.model_year - 0.25) - 1
   
    #Dealer Level Loss DLL
    dla_df =dla_df.rename(columns={"valid_vintage": "book_vintage"})
    dla_df = dla_df.replace("current","2025 Q2")
    #ula_df_total = ula_df_total.replace("2025 Q1","current")
    #ula_df_total = ula_df_total.replace("2024 Q4","current")
    ula_df_total = pd.merge(ula_df_total, dla_df, how="left", on=['dealer_number','book_vintage'])
    ula_df_total.loc[ula_df_total.frni_flag==1,'pricing_scalar'] = ula_df_total.loc[ula_df_total.frni_flag==1,'pricing_scalar'].fillna(1.1)
    ula_df_total['pricing_scalar'] = ula_df_total['pricing_scalar'].fillna(1)

    
    rra_df_total['yob'] = expected_years_on_book # df['MOB'] / 12    
    rra_df_total['impound_prob'] = impound_probability
    rra_df_total.car_make = rra_df_total.car_make.str.upper()
    rra_df_total['car_class_only'] = np.select([rra_df_total.vehicle_class.str.contains('Pickup', na=False),
                                          rra_df_total.vehicle_class.str.contains('Sporty', na=False),
                                          rra_df_total.vehicle_class.str.contains('Small.*Car', na=False),
                                          rra_df_total.vehicle_class.str.contains('Car', na=False),
                                          rra_df_total.vehicle_class.str.contains('Small.*SUV', na=False),
                                          rra_df_total.vehicle_class.str.contains('Large.*SUV', na=False),
                                          rra_df_total.vehicle_class.str.contains('Minivan', na=False),
                                          rra_df_total.vehicle_class.str.contains(' Van', na=False)],
                                         ['Truck', 'Sporty', 'Compact', 'Car', 'SUV Small', 'SUV Large', 'Minivan', 'Work Van'],
                                         default='Other')
    rra_df_total.fuel_type = np.select([rra_df_total.fuel_type.str.contains('Electric|Plug|EV', na=False),
                                        rra_df_total.fuel_type.str.contains('Hyb', na=False),
                                        ((rra_df_total.fuel_type=='CNG')|(rra_df_total.fuel_type=='LPG')),
                                        ((rra_df_total.fuel_type=='null')|(rra_df_total.fuel_type==None))],
                                 ['EV', 'Hybrid', 'Gas', 'Gas'],
                                 default=rra_df_total.fuel_type)
    rra_df_total['car_lux'] = np.where((rra_df_total.vehicle_class.str.contains('Luxury', na=False)), 'Luxury', 'Standard')
    rra_df_total['car_japanese'] = np.where(rra_df_total.car_make.isin(['HONDA', 'ACURA', 'ISUZU', 'MAZDA', 'MITSUBISHI', 'SUZUKI', 'TOYOTA', 'LEXUS', 'SCION', 'TOYOYA']),
                                     'Japanese', 'Other')
#     rra_df_total['job_clean'] = # TODO: Part of new recovery model
    warnings.filterwarnings("ignore", category=UserWarning)
    rra_df_total.job_company = rra_df_total.job_company.fillna('not provided')
    rra_df_total['driver_flag'] = np.where(rra_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
                                     1, 0)
    warnings.filterwarnings("default", category=UserWarning)
    rra_df_total['mileage'] = rra_df_total.mileage_orig / 1000
    rra_df_total['r_mmi'] = mmi_standard_increase ** rra_df_total.yob
    # Handle NA values
    ula_df_total = ula_df_total.dropna(subset=['sale_price', 'cd_model_score', 'pti', 'lob'])
    ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
    ula_df_total.employment = ula_df_total.employment.fillna('not seasonal or waiter')
    ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
    ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)
    # Fill flags (non_fld_non_kmx, non_fld_an_stg_frn)
    # non_fld_non_kmx <- app_attr$lob %in% c("AN", "STG", "FRN", "MCY", "FLD", "ENT") & dealer_attr$online_flag == 0 & dealer_attr$non_fld_fixed == 0
    ula_df_total['dealer_filter_a'] = ula_df_total.lob.isin({'AN', 'STG', 'FRN', 'MCY', 'FLD', 'ENT'}) # In the pricing code, this is currently labeled as "non_fld_non_kmx", even though FLD is included.
    # dealer_attr$non_fld_fixed <- ifelse(!is.na(dealer_attr$specialty_dealer) && (tolower(dealer_attr$specialty_dealer) %in% c('non-fld fixed 1') | (tolower(dealer_attr$specialty_dealer) %in% c("non-fld fixed 2") && app_attr$a_b_call == "A")), 1, 0)

    # non_fld_an_stg_frn <- app_attr$lob %in% c("AN", "STG", "FRN", "FLD", "ENT") & dealer_attr$online_flag == 0 & dealer_attr$non_fld_fixed == 0
    ula_df_total['dealer_filter_b'] = ula_df_total.lob.isin({'AN', 'STG', 'FRN', 'FLD', 'ENT'}) # In the pricing code, this is currently labeled as "non_fld_an_stg_frn", even though it's actually non-KMX, non-MCY.
    ##---Populate Flags---#
    # NonKMX Flags
    ula_df_total['ent_fld_flag'] = (ula_df_total.lob=='ENT') | (ula_df_total.lob=='FLD')
    ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500) & (ula_df_total.lob != 'MCY')
    ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
    ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY')
    ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.3) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
    ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
    ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally')
    ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.lob != 'MCY') & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total.book_date >= '2022-07-01')
    ula_df_total['mcy_low_mileage_flag'] = (ula_df_total.lob == 'MCY') & (ula_df_total.cd_model_score > 140) & (ula_df_total.mileage <= 20000) & (ula_df_total.vehicle_age <= 10)
    ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0,6])
    ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1,6))
    ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
    ula_df_total['chime_flag'] = ula_df_total.chime_indicator # | ula_df_total.secured_credit_card
    ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
    ula_df_total['seasonal_employment_flag'] = ula_df_total.employment == 'seasonal'
    ula_df_total['waiter_employment_flag'] = ula_df_total.employment == 'waiter'
    ula_df_total['high_cash_down_echopark_flag'] = False # Removed, only ever implemented as a test
    ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
    ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
    ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
    ula_df_total['null_fico_null_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & ((ula_df_total.vantage_score < 300) | (ula_df_total.vantage_score > 850))
    ula_df_total['pricing_change_flag'] = ula_df_total.book_date>= '2024-10-01'
    # KMX Flags
    ula_df_total['job_time_flag'] = ula_df_total.employed_months < 6
    ula_df_total['low_fico_flag'] = (ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 450)
    ula_df_total['high_model_score_flag'] = ula_df_total.cd_model_score >= 146
    ula_df_total['low_vantage_flag'] = (ula_df_total.fico_score < 300) & (ula_df_total.vantage_score > 4) & (ula_df_total.vantage_score < 450)
    ula_df_total['louisiana_flag'] = ula_df_total.state == 'LA'
    ula_df_total['normal_pti_flag'] = ula_df_total.pti <= 0.2
    ula_df_total['high_pti_tier_1_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.pti <= 0.25)
    ula_df_total['high_pti_tier_2_flag'] = (ula_df_total.pti > 0.25) & (ula_df_total.pti <= 0.35)
    ula_df_total['high_pti_tier_3_flag'] = ula_df_total.pti > 0.35
    ula_df_total['existing_dq_flag'] = ula_df_total.existing_dq_count > 0
    #Existing DQ Flag already in df
    ula_df_total['kmx_toyho_flag'] = ula_df_total.cd_model_score>=130 & ula_df_total.make.isin({'HON', 'TOY'})
    ula_df_total['kmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines >= 0.14
    ula_df_total['low_bureau_flag'] = ((ula_df_total.fico_score > 300) & (ula_df_total.fico_score < 475)) | (ula_df_total.vantage_score > 4) & ((ula_df_total.vantage_score < 450))
    ula_df_total['soft_pull_flag'] = ula_df_total.pull_type == 'softpull'
    ula_df_total['cd_perc_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1)
    ula_df_total['open_tl_flag'] = ula_df_total.open_tl = 0 
    ula_df_total['narrowed_soft_pull_flag'] = (ula_df_total.sale_price <= 18000) & (ula_df_total.cash_down / ula_df_total.sale_price <= 0.1) & ula_df_total.soft_pull_flag & (~ula_df_total.job_time_flag)
    # Add in vintages
    if granularity == 'q':
        rra_df_total['vintage'] = rra_df_total.book_date.str[:4] + ' Q' + ((rra_df_total.book_date.str[5:7].astype(int) - 1) // 3 + 1).astype(str)
        ula_df_total['vintage'] = ula_df_total.book_date.str[:4] + ' Q' + ((ula_df_total.book_date.str[5:7].astype(int) - 1) // 3 + 1).astype(str)
    elif granularity == 'm':
        rra_df_total['vintage'] = rra_df_total.book_date.str[:4] + ' M' + rra_df_total.book_date.str[5:7]
        ula_df_total['vintage'] = ula_df_total.book_date.str[:4] + ' M' + ula_df_total.book_date.str[5:7]
    elif granularity == 'w':
        rra_df_total['vintage'] = rra_df_total.book_date.str[:4] + '-' + rra_df_total.book_week.str.zfill(2)
        ula_df_total['vintage'] = ula_df_total.book_date.str[:4] + '-' + ula_df_total.book_week.str.zfill(2)
    # Many customers have multiple jobs. This creates many duplicates. Condense the flags down:
    ula_df_total = ula_df_total.drop_duplicates().copy()
    ula_df_total['employment_type_code'] = ula_df_total.seasonal_employment_flag * 1 + ula_df_total.waiter_employment_flag * 8
    combined_employment_code_df = ula_df_total.groupby('account_number').employment_type_code.sum().reset_index()
    ula_df_total = ula_df_total.drop(columns='employment_type_code').merge(combined_employment_code_df, on='account_number')
    ula_df_total.seasonal_employment_flag = (ula_df_total.employment_type_code == 1) | (ula_df_total.employment_type_code == 9)
    ula_df_total.waiter_employment_flag = (ula_df_total.employment_type_code == 8) | (ula_df_total.employment_type_code == 9)
    ula_df_total = ula_df_total.drop(columns='employment').drop_duplicates()
    # Do the same for rra_df_total
    combined_driver_flag_df = rra_df_total.groupby('account_number').driver_flag.max().reset_index()
    rra_df_total = rra_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
    rra_df_total = rra_df_total.drop(columns='job_company').drop_duplicates()
    store_pickle((ula_df_total, rra_df_total), '(ula_df_total, rra_df_total)_refined_pickle')
else:
    ula_df_total, rra_df_total = get_pickle('(ula_df_total, rra_df_total)_refined_pickle')

In [16]:
#rra_df_total.head()

In [28]:
"""
Wrapper for the RAGU Score calculations.
Baselines and other standard assumptions are established here.
Also groups by POS once everything is done, though we don't really look at POS.
"""
all_df_no_dropout = None
# leave_out_list = ['None', 'Previous ACA chargeoff', 'Small amount financed', 'Zero cash down', 'High mileage vehicle', 
#                    'High PTI', 'Car make', 'Theft risk', 'MCY high model score, low mileage', 'Weekday/weekend decision', 
#                    'Secured credit (Chime, etc.)', 'Employment type', 'High cash down EchoPark', 'Authorized tradelines', 
#                    'Clip', 'Vehicle Age', 'Dealer Level (Non-KMX)', 'Mileage/age', 'Japanese', 'Vehicle class', 'Fuel type', 
#                    'Luxury classification', 'LOB', 'Driver flag', 'Impound probability', 'Job time', 'Low FICO', 'Low Vantage',
#                    'Louisiana', 'Existing DQ', 'Null FICO with Vantage', 'Soft pull']
leave_out_list = ['None']
full_df_list_dict_dict = dict()
for leave_out in leave_out_list:
    print(leave_out)
    # Run all vintages
    available_vintages_dict = {'non_kmxent': list(ms_df[ms_df.lob.isin(['FRN', 'STG', 'AN', 'FLD'])].vintage.unique()), 
                          'ENT': list(ms_df[ms_df.lob == 'ENT'].vintage.unique()), 
                          'KMX': list(ms_df[ms_df.lob == 'KMX'].vintage.unique())}
    full_df_list_dict = {'non_kmxent': [], 'ENT': [], 'KMX': []}
    vintages_dict = {'non_kmxent': [], 'ENT': [], 'KMX': []}
    baseline_ltv_dict = {'non_kmxent': 1.94, 'ENT': 1.45, 'KMX': 1.5863}
    baseline_recovery_unadjusted_pct_dict = {'non_kmxent': 0.4308, 'ENT': 0.46, 'KMX': 0.3546}
    baseline_recovery_100_ltv_pct_dict = {'non_kmxent': 0.4308, 'ENT': 0.456, 'KMX': 0.3546}
    baseline_recovery_pct_dict = {'non_kmxent': 0.4308, 'ENT': 0.456, 'KMX': 0.3546}
    mmi_standard_increase = 1.03
    expected_years_on_book = 2
    impound_probability = 0.15
    mean_unit_loss = 0.5
    unit_loss_to_model_score = 0.02
    skip_rate = 0.75 # Not actually skip rate, should be % of CO sold
    ltv_realization_dollar = 0
    for lob, lob_type in zip([('FRN', 'AN', 'STG', 'FLD'), 'ENT', 'KMX'], 
                             ['non_kmxent', 'ENT', 'KMX']):
        available_vintages = available_vintages_dict[lob_type]
        baseline_ltv = baseline_ltv_dict[lob_type]
        baseline_recovery_pct = baseline_recovery_pct_dict[lob_type]
        baseline_recovery_unadjusted_pct = baseline_recovery_unadjusted_pct_dict[lob_type]
        baseline_recovery_100_ltv_pct = baseline_recovery_100_ltv_pct_dict[lob_type]
        for year in range(2016, 2026): #revert to original, change year to 2016
            if granularity == 'q':
                sub_year_list = range(1, 5)
            elif granularity == 'm': 
                sub_year_list = range(1, 13) 
            else:
                sub_year_list = range(1,53)
            for sub_year in sub_year_list:
                if year==2025 and ((granularity=='q' and sub_year > max_quarter) or (granularity=='m' and sub_year > max_month) or(granularity=='w' and sub_year > max_week)):
                    continue
                if granularity == 'm' or granularity == 'w':
                    sub_year = str(sub_year).zfill(2)
                if granularity == 'q':
                    vintage = f"{year} Q{sub_year}"
                elif granularity == 'm':
                    vintage = f"{year} M{sub_year}"
                else:
                    vintage = f"{year}-{sub_year}"
                if vintage in ('2022 M02', '2023 M11','2022-05','2022-06','2022-07','2022-08','2022-09','2023-44','2023-45','2023-46','2023-47','2023-48') and lob=='KMX':
                    continue
                if vintage in ('2023-14','2023-15') and lob_type in ('non_kmxent','ENT'):
                    continue
                if vintage in vintages_dict[lob_type] or vintage not in available_vintages:
                    continue
                print(vintage, lob)
                full_df = get_ragu_score(vintage, lob, lob_type, ula_df_total, rra_df_total, leave_out=leave_out)
                if type(lob)==str:
                    if lob in full_df_list_dict.keys():
                        full_df_list_dict[lob].append(full_df)
                        vintages_dict[lob].append(vintage)
                    else:
                        full_df_list_dict[lob] = [full_df]
                        vintages_dict[lob] = [vintage]
                else:
                    if lob_type in full_df_list_dict.keys():
                        full_df_list_dict[lob_type].append(full_df)
                        vintages_dict[lob_type].append(vintage)
                    else:
                        full_df_list_dict[lob_type] = [full_df]
                        vintages_dict[lob_type] = [vintage]
    all_df = pd.DataFrame(columns=list(full_df_list_dict[lob if type(lob)==str else lob_type][0].columns)) # lob_type doesn't matter, so long as it's valid
    for lob_type in ['non_kmxent', 'ENT', 'KMX']: # No need to change this for different LOBs, we just skip empty LOBs
        for full_df, vintage in zip(full_df_list_dict[lob_type], vintages_dict[lob_type]):
            concat_df = full_df.copy()
            all_df = pd.concat([all_df, concat_df])
    # Add POS
    if len(all_df.reset_index(names='lob').lob.unique()) > 1:
        individual_lob_df = all_df.reset_index(names='lob').sort_values(['vintage', 'lob']).reset_index(drop=True)
        individual_lob_df = individual_lob_df[individual_lob_df.lob==individual_lob_df.lob.str.upper()].rename(columns={'amt_financed_x':'amt_financed'})
        pos_df = individual_lob_df.groupby('vintage').apply(weighted_average_and_sum, ['ltv', 'ms_original' ,'ms_gla', 'ms_exclude_ltv', 'ms_100_ltv', 'ragu_score']).reset_index()
        pos_df['lob'] = 'POS'
        pos_df = pos_df.rename(columns={'amt_financed':'amt_financed_x'})
        all_df = pd.concat([all_df, pos_df.set_index('lob')])
    if leave_out=='None':
        all_df_no_dropout = all_df.copy()
    full_df_list_dict_dict[leave_out] = full_df_list_dict

None
2016 Q1 ('FRN', 'AN', 'STG', 'FLD')
coef_intercept
coef_yob
coef_mlg_x_age
coef_japanese
coef_class
coef_fuel
coef_lux
coef_lux
coef_lob
coef_driver
coef_impound
2016 Q2 ('FRN', 'AN', 'STG', 'FLD')


C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\2796428171.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_mix_df = bb_populated_df.groupby('lob').apply(weighted_average_and_sum, ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'])


coef_intercept
coef_yob
coef_mlg_x_age
coef_japanese
coef_class
coef_fuel
coef_lux
coef_lux
coef_lob
coef_driver
coef_impound
2016 Q3 ('FRN', 'AN', 'STG', 'FLD')


C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\2796428171.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_mix_df = bb_populated_df.groupby('lob').apply(weighted_average_and_sum, ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'])


coef_intercept
coef_yob
coef_mlg_x_age
coef_japanese
coef_class
coef_fuel
coef_lux
coef_lux
coef_lob
coef_driver
coef_impound
2016 Q4 ('FRN', 'AN', 'STG', 'FLD')


...

C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\3942498885.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1. 1. 1. ... 1. 1. 1.]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.2 * ula_df.job_time_flag
C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\2796428171.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_mix_df = bb_populated_df.groupby('lob').apply(weighted_average_and_sum, ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'])


2025 Q1 KMX


C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\3942498885.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.  1.  1.  ... 1.2 1.  1.2]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.2 * ula_df.job_time_flag
C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\2796428171.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_mix_df = bb_populated_df.groupby('lob').apply(weighted_average_and_sum, ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'])


2025 Q2 KMX


C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\3942498885.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[1.  1.  1.  ... 1.  1.  1.2]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  ula_df.loc[~ula_df.mtn_3_1_flag,'loss_multiplier'] *= 1 + 0.2 * ula_df.job_time_flag
C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\2796428171.py:84: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped_mix_df = bb_populated_df.groupby('lob').apply(weighted_average_and_sum, ['loss_multiplier', 'recovery_unadjusted_multiplier', 'ltv', 'bbvalue'])
C:\Users\olivia.he\AppData\Lo

In [29]:
# Uncomment the DataFrame assignment appropriate for the current bucket
# all_df_a = all_df.copy()
all_df_b = all_df.copy()
# all_df_c = all_df.copy()
#all_df_d = all_df.copy()

In [23]:
all_df_b

,amt_financed_x,loss_multiplier,recovery_unadjusted_multiplier,ltv,bbvalue,ltv_realization_factor,recovery_100_ltv_multiplier,recovery_multiplier,vintage,model_score,amt_financed_y,est_unit_loss,unit_loss_score,ms_original,baselined_recovery,baselined_unadjusted_recovery,baselined_100_ltv_recovery,ms_gla,ms_exclude_ltv,ms_100_ltv,ragu_score
AN,2.471022e+07,1.018401,0.419156,1.938686,9549.343347,0.0,0.0,0.0,2016 Q1,128.128468,28184209.70,0.5,127.668445,128.128468,0.0,0.972971,0.0,127.668445,126.945232,127.668445,127.668445
FLD,1.290477e+06,1.055416,0.447375,1.820504,9369.348017,0.0,0.0,0.0,2016 Q1,133.529120,376818.15,0.5,132.143732,133.529120,0.0,1.038475,0.0,132.143732,133.281025,132.143732,132.143732
FRN,1.580155e+07,1.090586,0.398908,2.131764,8347.489120,0.0,0.0,0.0,2016 Q1,128.035821,12320290.10,0.5,125.771167,128.035821,0.0,0.925969,0.0,125.771167,123.914070,125.771167,125.771167
STG,2.295842e+07,1.006955,0.428578,1.964594,9066.471885,0.0,0.0,0.0,2016 Q1,127.531566,31677088.16,0.5,127.357686,127.531566,0.0,0.994842,0.0,127.357686,127.216912,127.357686,127.357686
non_kmxent,6.476067e+07,NaN,0.418118,1.992626,NaN,NaN,0.0,0.0,2016 Q1,NaN,NaN,0.5,127.184521,128.001871,0.0,0.970561,0.0,127.184521,126.401773,127.184521,127.184521
AN,1.958086e+07,1.025774,0.425723,1.860507,10786.213428,0.0,0.0,0.0,2016 Q2,129.390639,21965902.75,0.5,128.746294,129.390639,0.0,0.988215,0.0,128.746294,128.423332,128.746294,128.746294
FLD,6.513040e+05,0.943885,0.427211,1.856784,8285.410669,0.0,0.0,0.0,2016 Q2,132.511575,191616.25,0.5,133.914448,132.511575,0.0,0.991669,0.0,133.914448,133.676131,133.914448,133.914448
FRN,1.628378e+07,1.098470,0.394773,2.142770,8313.275469,0.0,0.0,0.0,2016 Q2,130.133407,11744802.56,0.5,127.671648,130.133407,0.0,0.916372,0.0,127.671648,125.564164,127.671648,127.671648
STG,1.771300e+07,1.001081,0.432397,1.947859,9761.163769,0.0,0.0,0.0,2016 Q2,129.231593,25141015.20,0.5,129.204556,129.231593,0.0,1.003706,0.0,129.204556,129.308078,129.204556,129.204556
non_kmxent,5.422894e+07,NaN,0.418627,1.973752,NaN,NaN,0.0,0.0,2016 Q2,NaN,NaN,0.5,128.635356,129.599209,0.0,0.971744,0.0,128.635356,127.874546,128.635356,128.635356


In [25]:
#all_df_a.to_csv('all_df_a_.csv')

# Once everything's been run, put it all together and save it

In [31]:
all_df_a['pti_bucket'] = 'A'
all_df_b['pti_bucket'] = 'B'
# all_df_c['apr_bucket'] = 'C'
# all_df_d['apr_bucket'] = 'D'
all_df = pd.concat([all_df_a, all_df_b])#, all_df_c, all_df_d])
all_df = all_df.drop(columns=['loss_multiplier','recovery_unadjusted_multiplier','bbvalue','ltv_realization_factor','recovery_100_ltv_multiplier','recovery_multiplier','amt_financed_y','est_unit_loss','unit_loss_score','ms_original','baselined_recovery',	'baselined_unadjusted_recovery','baselined_100_ltv_recovery','ms_100_ltv','ragu_score'])

all_df.to_csv('low_cd_ragu_score.csv')

# After having the excel sheet

In [33]:
df = pd.read_excel("C:/ACA/nonkmx/ragu/Pool Ragu 2025.07 - low cd/low_cd_ragu_score.xlsx", sheet_name='low_cd_ragu_score')
df

,Unnamed: 0,amt_financed_x,ltv,vintage,model_score,ms_gla,ms_exclude_ltv,pti_bucket,baseline_ltv,ragu score
0,AN,1.608958e+06,2.252758,2016 Q1,126.608903,126.097489,124.993811,A,1.94,122.633643
1,FRN,2.183221e+06,2.454969,2016 Q1,127.354567,124.879815,123.234450,A,1.94,119.668431
2,STG,3.189039e+06,2.201304,2016 Q1,126.243028,126.168794,126.052852,A,1.94,124.034879
3,non_kmxent,6.981219e+06,2.292491,2016 Q1,NaN,125.749261,124.905017,A,NaN,NaN
4,AN,1.700359e+06,2.199685,2016 Q2,129.461864,128.515445,127.815087,A,1.94,125.808144
5,FLD,1.118672e+05,2.209960,2016 Q2,129.000000,130.349794,130.017338,A,1.45,124.171388
6,FRN,3.771421e+06,2.545851,2016 Q2,129.825875,126.432746,123.914167,A,1.94,119.868580
7,STG,3.184340e+06,2.196058,2016 Q2,127.984451,127.732083,127.600574,A,1.94,125.618391
8,non_kmxent,8.767987e+06,2.347397,2016 Q2,NaN,127.358506,126.023711,A,NaN,NaN
9,AN,2.440687e+06,2.348070,2016 Q3,128.542092,126.846658,125.682335,A,1.94,122.727913


In [ ]:
df = df[df['Unnamed: 0'].isin(['AN', 'STG', 'FRN', 'FLD', 'ENT'])]
df_vintage = df[df['vintage'].isin(['2022 Q3', '2022 Q4', '2023 Q1', '2023 Q2', '2023 Q3', '2023 Q4', '2024 Q1', '2024 Q2'])]
df_vintage = df_vintage[df_vintage['Unnamed: 0'] == 'ENT']

In [42]:
def weighted_avg(group):
    return (group['ragu score'] * group['amt_financed_x']).sum() / group['amt_financed_x'].sum()

weighted_df = df_vintage.groupby(['vintage', 'pti_bucket']).apply(weighted_avg).reset_index(name='weighted_ragu_score')
weighted_df

C:\Users\olivia.he\AppData\Local\Temp\ipykernel_26004\2570908402.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weighted_df = df_vintage.groupby(['vintage', 'pti_bucket']).apply(weighted_avg).reset_index(name='weighted_ragu_score')


,vintage,pti_bucket,weighted_ragu_score
0,2022 Q3,A,129.722118
1,2022 Q3,B,136.826079
2,2022 Q4,A,128.257374
3,2022 Q4,B,136.138968
4,2023 Q1,A,129.064802
5,2023 Q1,B,136.269225
6,2023 Q2,A,131.478800
7,2023 Q2,B,138.082596
8,2023 Q3,A,132.425823
9,2023 Q3,B,139.624374
